# ADAPT-BIO: Anticipatory Dynamic Attention Pruning Topology
### Biologically-Inspired Sparse Attention via Slime Mold Anticipation, RNA-Style Mask Re-Editing, and Starling Flock Topology

**Author:** Kritika Benjwal  
**Notebook:** 2 of 2 — GPT2 Tokenizer Experiments (WikiText-2 & WikiText-103)  
**Status:** Consolidated research notebook — GPT2 tokenizer experiments, T=512 scaling, k-sweep, RNA sweep, and cross-tokenizer summary

---

## Overview

ADAPT-BIO introduces **SOMA** (Slime mold / Octopus RNA / starling-Murmuration Attention), a sparse attention mechanism for transformers that fuses three biologically-inspired signals:

1. **Movement Pruning (Slime Mold Anticipation)** — tracks attention weight movement during an initial anticipation window to discover an importance signal before any pruning begins.
2. **RNA-Style Mask Refinement** — periodically re-edits the sparsity mask every Δ steps based on the accumulated movement signal.
3. **Starling Topology Constraint** — enforces a fixed top-k=7 neighbor sparsity pattern per query, inspired by local-interaction rules in starling murmurations.

## What this notebook contains

| Section | Contents |
|---|---|
| 1 | Environment setup — loads NB1 (BERT) results.json from Kaggle input dataset |
| 2 | SOMA core components (redefined for standalone operation) |
| 3 | ADAPT-BIO transformer architecture (redefined for standalone operation) |
| 4 | Data loading — BERT tokenizer base + GPT2 tokenizer datasets |
| 5 | Training & benchmarking utilities |
| 23 | GPT2 tokenizer setup — WikiText-2 & WikiText-103 datasets |
| 24 | GPT2 main results — Dense vs ADAPT-BIO, d_model=128 & d_model=256 |
| 25 | GPT2 T=512 scaling experiment — WikiText-2 |
| 26 | GPT2 T=512 scaling experiment — WikiText-103 |
| 27 | GPT2 k-sweep & RNA interval sweep |
| 28 | Cross-tokenizer summary — BERT vs GPT2 |

> **Note:** This notebook requires Notebook 1 (BERT) results.json as an input dataset.  
> After NB1 finishes, attach its output `results.json` as a Kaggle dataset input to this notebook.  
> Cell 4 copies it to `/kaggle/working/results.json` so all BERT keys are available for the cross-tokenizer summary.  
> All results are saved incrementally — the notebook is fully resumable via `if 'key' not in results` guards.

## 1. Environment Setup

This cell clones the ADAPT-BIO repository and installs dependencies.

> ⚠️ **Security note:** Your GitHub token must come from **Kaggle Secrets**, not be hardcoded in the notebook. Before running this cell for the first time:
> 1. Revoke the old exposed token at https://github.com/settings/tokens (if you haven't already)
> 2. Generate a new fine-scoped token
> 3. In this notebook: **Add-ons → Secrets → Add Secret**, name it `GITHUB_TOKEN`, paste the new token
>
> The cell below reads it via `UserSecretsClient` — it will never appear in plaintext, in version history, or in any exported `.ipynb`.

In [1]:
!git config --global user.name "Kritika Benjwal"
!git config --global user.email "ananya.benjwal@gmail.com"

In [2]:
import os, shutil

RESULTS_DEST = '/kaggle/working/results.json'  # writable location

def find_nb1_results():
    search_root = '/kaggle/input/'
    if not os.path.exists(search_root):
        return None
    for dirpath, dirnames, filenames in os.walk(search_root):
        for fname in filenames:
            if fname.lower() in ('results.json', 'result.json'):  # handle both
                return os.path.join(dirpath, fname)
    return None

nb1_path = find_nb1_results()
if nb1_path:
    shutil.copy(nb1_path, RESULTS_DEST)
    print(f"✅ Copied NB1 results.json from: {nb1_path}")
    print(f"   → Destination: {RESULTS_DEST}")
else:
    print("⚠️  NB1 results.json not found in /kaggle/input/")

✅ Copied NB1 results.json from: /kaggle/input/datasets/sarthakgupta1801/result/result.json
   → Destination: /kaggle/working/results.json


In [3]:
import os, sys, time

# ── GitHub token from Kaggle Secrets (never hardcoded) ──────────────────────
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")

!git clone https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git
%cd /kaggle/working/adapt-bio
!pip install -q einops omegaconf datasets transformers

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("PyTorch:", torch.__version__)

# Make local repo importable
sys.path.insert(0, '/kaggle/working/adapt-bio/src')

# Results store — every later section appends here and saves after each run
import json
RESULTS_PATH = '/kaggle/working/results.json'
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        results = json.load(f)
    print(f"Loaded existing results.json with keys: {list(results.keys())}")
else:
    results = {}
    print("Starting fresh results.json")

def save_results():
    with open(RESULTS_PATH, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f"✅ results.json saved ({list(results.keys())})")

Cloning into 'adapt-bio'...
remote: Enumerating objects: 602, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 602 (delta 83), reused 181 (delta 46), pack-reused 381 (from 1)
Receiving objects: 100% (602/602), 8.62 MiB | 25.58 MiB/s, done.
Resolving deltas: 100% (242/242), done.
/kaggle/working/adapt-bio
CUDA available: True
Device: Tesla T4
PyTorch: 2.10.0+cu128
Loaded existing results.json with keys: ['main_wt2_128', 'main_wt2_256', 'k_sweep', 'pareto_analysis', 'rna_sweep', 'component_ablation', 't512_experiment', 'benchmarks', 'main_wt103_128', 'main_wt103_256', 'significance_tests', 'wt103_k_sweep', 'wt103_pareto_analysis', 'wt103_rna_sweep', 'wt103_component_ablation', 'wt103_t512_experiment', 'wt103_benchmarks']


## 2. SOMA Core Components

All three bio-inspired modules are defined inline here (no `src/` import needed for standalone runs).  
Key fix vs earlier notebooks: `SOMAMask.forward` stays **dense** during the anticipation window (`step < anticipation_steps`) and only starts emitting a sparse mask after the window closes.

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ── Signal 3 (listed first because Signal 1 depends on it) ──────────────────

class StarlingTopologyConstraint(nn.Module):
    """
    Starling murmuration topology: each token attends to exactly its k
    most movement-active neighbours (k=7 by default, per Ballerini et al. 2008).
    """
    def __init__(self, k: int = 7):
        super().__init__()
        self.k = k

    def apply(self, movement_scores: torch.Tensor) -> torch.Tensor:
        """
        Args:
            movement_scores: (H, T, T)  – accumulated movement signal
        Returns:
            boolean mask:   (H, T, T)  – True for the top-k positions per row
        """
        k = min(self.k, movement_scores.shape[-1])
        _, topk_idx = movement_scores.topk(k, dim=-1)          # (H, T, k)
        mask = torch.zeros_like(movement_scores, dtype=torch.bool)
        mask.scatter_(-1, topk_idx, True)
        return mask


# ── Signal 1 ────────────────────────────────────────────────────────────────

class MovementPruningMask(nn.Module):
    """
    Slime-mould anticipation: accumulates step-wise attention-weight movement
    to discover which edges are gradient-sensitive.

    Two-phase accumulation (journal version — fixes Eq.1 in conference paper):

      Anticipation window (step ≤ S):
        A_h^(t) = Σ_{τ=1}^{t} |W_h^(τ) − W_h^(τ-1)|          (cumulative, Eq.1)

      Post-anticipation (step > S):
        A_h^(t) = (1 − α) · A_h^(t-1) + α · |W_h^(t) − W_h^(t-1)|   (EMA, Eq.1b)

    The EMA phase is the key journal addition over the conference paper:
    it keeps the movement signal live and responsive to the evolving gradient
    landscape, making each RNA re-edit a genuine revision rather than a
    deterministic replay of a frozen 100-step snapshot.
    α (ema_alpha) is a new hyperparameter; 0.05 was found optimal via
    informal sweep and is fixed for all reported experiments.
    """

    def __init__(self,
                 num_heads: int,
                 seq_len: int,
                 anticipation_steps: int,
                 ema_alpha: float = 0.05):
        super().__init__()
        self.anticipation_steps = anticipation_steps
        self.ema_alpha = ema_alpha
        self.step = 0
        self.register_buffer("movement_accum",
                             torch.zeros(num_heads, seq_len, seq_len))
        self.register_buffer("prev_weights",
                             torch.zeros(num_heads, seq_len, seq_len))

    def update(self, attn_weights: torch.Tensor) -> None:
        """
        Called once per forward pass with the attention signal
        averaged over the batch: shape (H, T, T).
        """
        w = attn_weights.detach()

        if self.step > 0:
            delta = (w - self.prev_weights).abs()

            if self.step <= self.anticipation_steps:
                # Cumulative sum — Eq. 1 (conference paper, also Eq. 1 journal)
                self.movement_accum.add_(delta)
            else:
                # EMA — Eq. 1b (journal addition)
                # Keeps signal live so RNA re-edits reflect current training state
                self.movement_accum.mul_(1.0 - self.ema_alpha).add_(
                    delta * self.ema_alpha
                )

        self.prev_weights.copy_(w)
        self.step += 1


# ── Signal 2 ────────────────────────────────────────────────────────────────

class RNAMaskRefinement(nn.Module):
    """
    Octopus-RNA self-editing: re-evaluates the sparsity mask every
    `update_interval` steps using the current (EMA-fresh) movement signal.
    """
    def __init__(self, update_interval: int = 500):
        super().__init__()
        self.update_interval  = update_interval
        self.last_update_step = 0

    def should_update(self, step: int) -> bool:
        return (step - self.last_update_step) >= self.update_interval

    def refine(self, current_mask, movement_signal, step, k=7):
        if not self.should_update(step):
            return current_mask
        new_mask = StarlingTopologyConstraint(k=k).apply(movement_signal)
        self.last_update_step = step
        return new_mask


# ── SOMA integration ─────────────────────────────────────────────────────────

class SOMAMask(nn.Module):
    """
    Self-Organising Mask Anticipation — combines all three biological signals.

    Timeline
    ────────
    steps 0 … anticipation_steps−1   : DENSE   (slime-mould sensing, cumulative Eq.1)
    steps anticipation_steps … Δ−1   : DENSE   (waiting for first RNA edit, EMA Eq.1b)
    step  Δ, 2Δ, 3Δ …               : SPARSE  (RNA re-edits with EMA-fresh signal)

    Ablation flags (Section 9):
    ┌──────────────────┬──────────┬────────────┬──────────────────────────────────┐
    │ Config           │ use_rna  │use_starling│ Behaviour                        │
    ├──────────────────┼──────────┼────────────┼──────────────────────────────────┤
    │ Full ADAPT-BIO   │ True     │ True       │ RNA + k=7 topology               │
    │ No-RNA           │ False    │ True       │ freeze once at S, k=7 topology   │
    │ No-Starling      │ True     │ False      │ RNA + k=T (dense topology)       │
    │ Movement-only    │ False    │ False      │ freeze once at S, raw top-k,     │
    │                  │          │            │ no bio topology or revision       │
    └──────────────────┴──────────┴────────────┴──────────────────────────────────┘
    """
    def __init__(self, num_heads, seq_len, k=7,
                 anticipation_steps=100, rna_update_interval=500,
                 ema_alpha=0.05,
                 use_rna=True, use_starling=True):
        super().__init__()
        self.anticipation_steps = anticipation_steps
        self.use_rna            = use_rna
        self.use_starling       = use_starling
        self.k                  = k

        self.movement = MovementPruningMask(
            num_heads, seq_len, anticipation_steps, ema_alpha=ema_alpha
        )
        self.rna = (RNAMaskRefinement(update_interval=rna_update_interval)
                    if use_rna else None)
        self.starling = StarlingTopologyConstraint(k=k) if use_starling else None

        self.register_buffer("current_mask",
                             torch.ones(num_heads, seq_len, seq_len,
                                        dtype=torch.bool))

    def forward(self, attn_weights: torch.Tensor, step: int) -> torch.Tensor:
        self.movement.update(attn_weights)

        if step < self.anticipation_steps:
            return self.current_mask

        signal = self.movement.movement_accum   # (H, T, T) — EMA-fresh post-S

        if self.use_rna:
            k_eff = self.k if self.use_starling else signal.shape[-1]
            self.current_mask = self.rna.refine(
                self.current_mask, signal, step=step, k=k_eff
            )
        elif self.use_starling and step == self.anticipation_steps:
            # No-RNA: freeze mask once at end of anticipation window
            self.current_mask = self.starling.apply(signal)
        elif not self.use_rna and not self.use_starling:
            # movement_only: freeze once using raw top-k, no bio topology label
            if step == self.anticipation_steps:
                self.current_mask = StarlingTopologyConstraint(k=self.k).apply(signal)

        return self.current_mask


# ── Smoke test ───────────────────────────────────────────────────────────────

def _smoke_test():
    print("Running architecture smoke tests ...\n")

    HEADS, T, ANTICI, K = 4, 32, 10, 7

    # Fix 1: EMA keeps movement_accum live after anticipation
    mm = MovementPruningMask(num_heads=HEADS, seq_len=T,
                             anticipation_steps=ANTICI, ema_alpha=0.05)
    snapshots = []
    for s in range(60):
        mm.update(torch.rand(HEADS, T, T))
        if s in (ANTICI - 1, ANTICI + 10, ANTICI + 20, ANTICI + 30):
            snapshots.append(mm.movement_accum.clone())

    same_after = all(torch.allclose(snapshots[0], sn) for sn in snapshots[1:])
    assert not same_after, "FAIL: movement_accum frozen after anticipation — EMA not applied."
    print("  Fix 1 ✅  movement_accum evolves after anticipation window (EMA active)\n")

    # Fix 2: StarlingTopologyConstraint gives exactly k per row
    sc   = StarlingTopologyConstraint(k=K)
    fake = torch.rand(HEADS, T, T)
    fake[0, 0, :K] = 0.99
    mask = sc.apply(fake)
    all_exactly_k = (mask.long().sum(dim=-1) == K).all().item()
    assert all_exactly_k, f"FAIL: some rows ≠ {K} edges"
    print(f"  Fix 2 ✅  every row has exactly {K} active edges\n")

    # Fix 3: movement_only produces sparse mask
    soma_mo = SOMAMask(num_heads=HEADS, seq_len=T, k=K,
                       anticipation_steps=ANTICI, rna_update_interval=5,
                       use_rna=False, use_starling=False)
    for s in range(ANTICI + 5):
        out = soma_mo(torch.rand(HEADS, T, T), step=s)
    sparsity_mv = 1.0 - out.float().mean().item()
    assert sparsity_mv > 0.5, f"FAIL: movement_only mask is dense (sparsity={sparsity_mv:.1%})"
    print(f"  Fix 3 ✅  movement_only sparsity={sparsity_mv:.1%}\n")

    # Fix 4: ema_alpha correctly passed through to MovementPruningMask
    soma_ema = SOMAMask(num_heads=HEADS, seq_len=T, k=K,
                        anticipation_steps=ANTICI, ema_alpha=0.10)
    assert soma_ema.movement.ema_alpha == 0.10, "FAIL: ema_alpha not forwarded"
    print("  Fix 4 ✅  ema_alpha correctly surfaced and forwarded\n")

    # Full SOMA end-to-end
    soma = SOMAMask(num_heads=HEADS, seq_len=T, k=K,
                    anticipation_steps=ANTICI, rna_update_interval=5)
    mask_snapshots = []
    for s in range(40):
        out = soma(torch.rand(HEADS, T, T), step=s)
        if s >= ANTICI:
            mask_snapshots.append(out.clone())

    assert not all(torch.equal(mask_snapshots[0], m) for m in mask_snapshots[1:]), \
        "FAIL: SOMA mask never changed — RNA re-editing is still a no-op."
    sparsity = 1.0 - mask_snapshots[-1].float().mean().item()
    print(f"  SOMA ✅  mask evolves | sparsity={sparsity:.1%}\n")
    print("All architecture smoke tests passed ✅")


_smoke_test()


# ── Commit corrected architecture to GitHub ───────────────────────────────
import os
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"fix: MovementPruningMask EMA + step-wise delta (Eq.1); '
          'StarlingTopologyConstraint exact-k via scatter_; '
          'SOMAMask movement_only branch (was silently dense)"')
os.system(f"cd /kaggle/working/adapt-bio && git push "
          f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main")
print("✅ Pushed to GitHub")

Running architecture smoke tests ...

  Fix 1 ✅  movement_accum evolves after anticipation window (EMA active)

  Fix 2 ✅  every row has exactly 7 active edges

  Fix 3 ✅  movement_only sparsity=78.1%

  Fix 4 ✅  ema_alpha correctly surfaced and forwarded

  SOMA ✅  mask evolves | sparsity=78.1%

All architecture smoke tests passed ✅
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
✅ Pushed to GitHub


Everything up-to-date


## 3. ADAPT-BIO Transformer Architecture

Self-contained architecture — no `src/` dependency needed.  
`SOMAAttention` wraps scaled dot-product attention and gates the weight matrix
with the SOMA boolean mask before softmax.  
The `use_rna` / `use_starling` flags are threaded all the way down to `SOMAMask`
so Section 9 ablations can swap configs without rewriting the model.

In [5]:
class SOMAAttention(nn.Module):
    """
    Multi-head self-attention gated by the SOMA sparsity mask.

    Architecture
    ────────────
    x (B, T, d_model)
      → dropout → QKV linear → (B, H, T, head_dim) × 3
      → scaled dot-product scores  attn  (B, H, T, T)
      → SOMA mask (H, T, T) derived from movement-accumulator signal
      → causal mask intersected with SOMA mask
      → diagonal fallback (self-attention always allowed to prevent NaN)
      → softmax → attention dropout → weighted sum over V
      → out_proj → dropout → (B, T, d_model)

    Causal masking
    ──────────────
    Without a causal mask, position i attends to i+1 and leaks the target token
    into the input. We intersect SOMA mask with a lower-triangular causal mask
    at every forward pass (training and evaluation).

    Diagonal fallback
    ─────────────────
    For early positions (e.g. i=0, only j=0 is causal), SOMA's k=7 columns may
    all be j>i, producing all-inf softmax → NaN. The diagonal fallback ensures
    at least one valid position per row. Self-attention is almost always among
    SOMA's top-k after the first few hundred steps, so this rarely fires.

    Dropout (CRITICAL FIX vs conference paper)
    ─────────────────────────────────────────
    The conference paper's ADAPTBIOTransformer had no dropout, making the
    comparison with FairDenseTransformer (dropout=0.3) unfair. This version
    adds embedding dropout and FF dropout matching the dense baseline exactly.
    The SOMA sparsity itself acts as structural regularisation, but explicit
    dropout is still added so the comparison is controlled.
    """
    def __init__(self, d_model, num_heads, seq_len, k=7,
                 anticipation_steps=100, rna_update_interval=500,
                 ema_alpha=0.05, dropout=0.0,
                 use_rna=True, use_starling=True):
        super().__init__()
        assert d_model % num_heads == 0, \
            f"d_model={d_model} must be divisible by num_heads={num_heads}"
        self.num_heads   = num_heads
        self.head_dim    = d_model // num_heads
        self.scale       = self.head_dim ** -0.5
        self.qkv         = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj    = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop   = nn.Dropout(dropout)
        self.resid_drop  = nn.Dropout(dropout)
        self.soma        = SOMAMask(
            num_heads=num_heads, seq_len=seq_len, k=k,
            anticipation_steps=anticipation_steps,
            rna_update_interval=rna_update_interval,
            ema_alpha=ema_alpha,
            use_rna=use_rna, use_starling=use_starling,
        )

    def forward(self, x: torch.Tensor, step: int = 0) -> torch.Tensor:
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        attn = (q @ k.transpose(-2, -1)) * self.scale   # (B, H, T, T)

        # ── Causal mask ───────────────────────────────────────────────────
        causal = torch.zeros(T, T, device=x.device)
        causal.masked_fill_(
            torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool),
                       diagonal=1),
            float('-inf')
        )                                                # (T, T)

        # ── SOMA sparsity mask ────────────────────────────────────────────
        soma_signal = attn.detach().mean(0)              # (H, T, T)
        soma_mask   = self.soma(soma_signal, step=step)  # (H, T, T)  True=keep

        # ── Diagonal fallback ─────────────────────────────────────────────
        diag    = torch.eye(T, dtype=torch.bool, device=x.device)
        allowed = (soma_mask | diag.unsqueeze(0))        # (H, T, T)

        # ── Apply combined mask ───────────────────────────────────────────
        attn = attn.masked_fill(~allowed.unsqueeze(0), float('-inf'))
        attn = attn + causal.unsqueeze(0).unsqueeze(0)  # broadcast (1,1,T,T)
        attn = F.softmax(attn, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.attn_drop(attn)

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(out))


class ADAPTBIOBlock(nn.Module):
    def __init__(self, d_model, num_heads, seq_len, k=7,
                 anticipation_steps=100, rna_update_interval=500,
                 ema_alpha=0.05, dropout=0.0,
                 use_rna=True, use_starling=True):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = SOMAAttention(d_model, num_heads, seq_len, k=k,
                                   anticipation_steps=anticipation_steps,
                                   rna_update_interval=rna_update_interval,
                                   ema_alpha=ema_alpha, dropout=dropout,
                                   use_rna=use_rna, use_starling=use_starling)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, step: int) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), step=step)
        x = x + self.ff(self.norm2(x))
        return x


class ADAPTBIOTransformer(nn.Module):
    """
    ADAPT-BIO autoregressive transformer.

    dropout=0.3 matches FairDenseTransformer's regularisation exactly,
    ensuring all PPL comparisons are controlled (CRITICAL FIX: conference
    paper's ADAPTBIOTransformer had no dropout anywhere).
    """
    def __init__(self, vocab_size, d_model, num_heads, num_layers, seq_len,
                 k=7, anticipation_steps=100, rna_update_interval=500,
                 ema_alpha=0.05, dropout=0.3,
                 use_rna=True, use_starling=True):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model)
        self.pos_enc = nn.Embedding(seq_len, d_model)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.ModuleList([
            ADAPTBIOBlock(d_model, num_heads, seq_len, k=k,
                          anticipation_steps=anticipation_steps,
                          rna_update_interval=rna_update_interval,
                          ema_alpha=ema_alpha, dropout=dropout,
                          use_rna=use_rna, use_starling=use_starling)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x: torch.Tensor, step: int = 0) -> torch.Tensor:
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        h    = self.drop(self.embed(x) + self.pos_enc(pos))
        for block in self.blocks:
            h = block(h, step=step)
        return self.head(self.norm(h))


class FairDenseTransformer(nn.Module):
    """
    Dense baseline. Causal mask passed as additive float mask (not bool)
    to avoid dtype-dependent behaviour in future PyTorch versions.
    """
    def __init__(self, vocab_size, d_model, num_heads, num_layers, seq_len, dropout=0.3):
        super().__init__()
        self.embed    = nn.Embedding(vocab_size, d_model)
        self.pos_enc  = nn.Embedding(seq_len, d_model)
        self.drop     = nn.Dropout(dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=d_model * 4, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor, step=None) -> torch.Tensor:
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        h    = self.drop(self.embed(x) + self.pos_enc(pos))

        causal_mask = torch.zeros(T, T, device=x.device)
        causal_mask.masked_fill_(
            torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool),
                       diagonal=1),
            float('-inf')
        )
        h = self.transformer(h, mask=causal_mask)
        return self.head(self.drop(h))


# ── Smoke tests ───────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Test 1: ADAPTBIOTransformer forward pass shape
_m = ADAPTBIOTransformer(vocab_size=1000, d_model=64, num_heads=4,
                          num_layers=2, seq_len=32, k=7, dropout=0.3).to(device)
_x = torch.randint(0, 1000, (2, 32), device=device)
_out = _m(_x, step=150)
assert _out.shape == (2, 32, 1000)
print(f"ADAPTBIOTransformer ✅  shape={_out.shape}  params={sum(p.numel() for p in _m.parameters()):,}")
del _m, _x, _out

# Test 2: FairDenseTransformer forward pass + causal mask dtype is float
_d = FairDenseTransformer(vocab_size=1000, d_model=64, num_heads=4,
                           num_layers=2, seq_len=32).to(device)
_x = torch.randint(0, 1000, (2, 32), device=device)
_out = _d(_x)
assert _out.shape == (2, 32, 1000)
print(f"FairDenseTransformer  ✅  shape={_out.shape}")
del _d, _x, _out

# Test 3: Dropout is actually present in ADAPTBIOTransformer
_m_drop = ADAPTBIOTransformer(vocab_size=100, d_model=32, num_heads=2,
                               num_layers=1, seq_len=16, k=3, dropout=0.5)
has_drop = any(isinstance(m, nn.Dropout) for m in _m_drop.modules())
assert has_drop, "FAIL: no Dropout found in ADAPTBIOTransformer — dropout fix not applied"
print("Dropout check         ✅  ADAPTBIOTransformer contains Dropout layers")
del _m_drop

# Test 4: Causal leak check — output at pos=0 must not change when pos=1 is perturbed
_attn = SOMAAttention(d_model=32, num_heads=2, seq_len=8, k=3,
                       anticipation_steps=2, rna_update_interval=100,
                       dropout=0.0).to(device)
_attn.eval()
_x_in = torch.randn(1, 8, 32, device=device)
_x_a  = _x_in.clone(); _x_a[0, 1, :] += 1.0
with torch.no_grad():
    _out_orig = _attn(_x_in, step=5)
    _out_perturbed = _attn(_x_a, step=5)
max_leak = (_out_perturbed[0, 0] - _out_orig[0, 0]).abs().max().item()
assert max_leak < 1e-5, f"FAIL: causal leakage detected Δ={max_leak:.4f}"
print(f"Causal leak check     ✅  max Δ at pos=0 from perturbing pos=1: {max_leak:.2e}\n")
del _attn, _x_in, _x_a, _out_orig, _out_perturbed

print("All architecture smoke tests passed ✅")

ADAPTBIOTransformer ✅  shape=torch.Size([2, 32, 1000])  params=229,632
FairDenseTransformer  ✅  shape=torch.Size([2, 32, 1000])
Dropout check         ✅  ADAPTBIOTransformer contains Dropout layers
Causal leak check     ✅  max Δ at pos=0 from perturbing pos=1: 0.00e+00

All architecture smoke tests passed ✅


## 4. Data Loading

WikiText-2 is loaded fully into memory (small — ~2M tokens).  
WikiText-103 uses a chunked `Dataset` with a token cap to avoid OOM on Kaggle.

In [6]:
import math, gc, time
from transformers import AutoTokenizer
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer.pad_token = tokenizer.eos_token if tokenizer.pad_token is None else tokenizer.pad_token
VOCAB_SIZE = tokenizer.vocab_size   # 30522
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Vocab : {VOCAB_SIZE}")
print(f"Device: {DEVICE}")


class WikiText2Dataset(Dataset):
    def __init__(self, split, seq_len=128):
        raw     = load_dataset("wikitext", "wikitext-2-raw-v1", split=split)
        all_ids = []
        for item in raw:
            t = item["text"].strip()
            if t:
                all_ids.extend(tokenizer.encode(t, add_special_tokens=False))
        n = (len(all_ids) // (seq_len + 1)) * (seq_len + 1)
        self.chunks = torch.tensor(all_ids[:n], dtype=torch.long).view(-1, seq_len + 1)
        print(f"  WikiText-2 [{split}]: {len(self.chunks)} chunks (seq_len={seq_len})")
    def __len__(self):        return len(self.chunks)
    def __getitem__(self, i): return self.chunks[i]


class WikiText103Dataset(Dataset):
    def __init__(self, split, n_chunks=20000, seq_len=128):
        raw     = load_dataset("wikitext", "wikitext-103-raw-v1",
                               split=split, trust_remote_code=True)
        all_ids = []
        for item in raw:
            t = item["text"].strip()
            if not t:
                continue
            all_ids.extend(tokenizer.encode(t, add_special_tokens=False))
            if len(all_ids) >= n_chunks * (seq_len + 1):
                break
        n = (len(all_ids) // (seq_len + 1)) * (seq_len + 1)
        self.chunks = torch.tensor(all_ids[:n], dtype=torch.long).view(-1, seq_len + 1)
        print(f"  WikiText-103 [{split}]: {len(self.chunks)} chunks (seq_len={seq_len})")
    def __len__(self):        return len(self.chunks)
    def __getitem__(self, i): return self.chunks[i]


def make_loaders(dataset_cls, seq_len=128, batch_size=32, **dataset_kwargs):
   
    tr = dataset_cls("train",      seq_len=seq_len, **dataset_kwargs)
    va = dataset_cls("validation", seq_len=seq_len, **dataset_kwargs)
    return (DataLoader(tr, batch_size=batch_size, shuffle=True,  drop_last=True),
            DataLoader(va, batch_size=batch_size, shuffle=False, drop_last=True))


SEQ_LEN    = 128
BATCH_SIZE = 32

print("\nLoading WikiText-2...")
wt2_train, wt2_val = make_loaders(WikiText2Dataset, seq_len=SEQ_LEN, batch_size=BATCH_SIZE)
print(f"  Train batches: {len(wt2_train)} | Val batches: {len(wt2_val)}")
print("Data loaders ready ✅")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab : 30522
Device: cuda

Loading WikiText-2...


...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (645 > 512). Running this sequence through the model will result in indexing errors


  WikiText-2 [train]: 17858 chunks (seq_len=128)
  WikiText-2 [validation]: 1850 chunks (seq_len=128)
  Train batches: 558 | Val batches: 57
Data loaders ready ✅


## 5. Training & Benchmarking Utilities

- `evaluate(model, loader)` — returns perplexity  
- `train_run(...)` — single training run, returns (final val PPL, wall time)  
- `theoretical_flops(k, seq_len)` — dense vs sparse attention FLOPs  
- `benchmark_timing(...)` — wall-clock train/inference time + peak GPU memory

In [7]:
import numpy as np
criterion = nn.CrossEntropyLoss()

# ── Sentinel step value for evaluation ───────────────────────────────────────
# Using float('inf') is cleaner than a magic integer: it will always be
# >= anticipation_steps regardless of how that hyperparameter is set.
# SOMAMask.forward checks `step < anticipation_steps`, so inf → sparse mode.
EVAL_STEP = 10**9   # int sentinel; torch doesn't accept float for step


def evaluate(model, loader, max_batches=200):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= max_batches: break
            batch    = batch.to(DEVICE)
            inp, tgt = batch[:, :-1], batch[:, 1:]
            logits   = model(inp, step=EVAL_STEP)   # always sparse during eval
            loss     = criterion(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
            total_loss   += loss.item() * tgt.numel()
            total_tokens += tgt.numel()
    return math.exp(total_loss / total_tokens)


def train_run(model, train_loader, val_loader, steps=10000,
              lr=3e-4, log_every=1000, label=""):
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    model.train()
    it = iter(train_loader)
    t0 = time.time()
    for step in range(steps):
        try:
            batch = next(it)
        except StopIteration:
            it = iter(train_loader)
            batch = next(it)
        batch    = batch.to(DEVICE)
        inp, tgt = batch[:, :-1], batch[:, 1:]
        logits   = model(inp, step=step)
        loss     = criterion(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        if step % log_every == 0:
            elapsed = time.time() - t0
            print(f"  [{label}] step {step:5d}/{steps} | "
                  f"loss={loss.item():.4f} | elapsed={elapsed:.0f}s")
    val_ppl = evaluate(model, val_loader)
    print(f"  [{label}] ✅ Final Val PPL = {val_ppl:.4f}  "
          f"(total {time.time()-t0:.0f}s)")
    return val_ppl, time.time() - t0


def theoretical_flops(k, seq_len):
    dense  = seq_len * seq_len
    sparse = seq_len * k
    return {
        "dense_attn_flops":  dense,
        "sparse_attn_flops": sparse,
        "flop_reduction":    1.0 - sparse / dense,
        "flops_ratio":       dense / sparse,
    }


def benchmark_timing(model, seq_len=128, batch_size=8,
                     n_warmup=20, n_train_steps=100, n_infer_steps=100, label=""):
   
    vocab = model.head.out_features
    x = torch.randint(0, vocab, (batch_size, seq_len), device=DEVICE)
    y = torch.randint(0, vocab, (batch_size, seq_len), device=DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)

    # ── Warmup ───────────────────────────────────────────────────────────────
    model.train()
    for step in range(n_warmup):
        logits = model(x, step=step)
        loss   = criterion(logits.reshape(-1, vocab), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats(DEVICE)

    # ── Training timing ───────────────────────────────────────────────────────
    model.train()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for step in range(n_train_steps):
        logits = model(x, step=n_warmup + step)
        loss   = criterion(logits.reshape(-1, vocab), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    train_ms = (time.time() - t0) / n_train_steps * 1000

    # ── Inference timing ──────────────────────────────────────────────────────
    model.eval()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for _ in range(n_infer_steps):
            _ = model(x, step=EVAL_STEP)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    infer_ms = (time.time() - t0) / n_infer_steps * 1000

    peak_mb = (torch.cuda.max_memory_allocated(DEVICE) / 1024**2
               if DEVICE.type == "cuda" else float("nan"))

    print(f"  [{label}] train {train_ms:.1f} ms/step | "
          f"infer {infer_ms:.1f} ms/step | "
          f"peak GPU {peak_mb:.1f} MB")
    return {"train_ms_per_step": round(train_ms, 2),
            "infer_ms_per_step": round(infer_ms, 2),
            "peak_gpu_mb":       round(peak_mb, 1)}


print("Utilities defined ✅")

Utilities defined ✅


In [8]:
# ── Dependency bridge ─────────────────────────────────────────────────────────
# These variables and functions are defined here rather than in their original
# NB1 cells (cells 13 and 15) which are not included in this notebook.
# Values are identical to NB1 — no changes to logic or hyperparameters.

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

STEPS = 10000
SEEDS = [42, 123, 7]


def is_pareto_optimal(k_candidate, sw, k_vals):
    """
    k_candidate is Pareto-optimal if no other k achieves both lower PPL
    AND equal or higher sparsity simultaneously.
    Exact copy of function defined in NB1 Cell 15.
    """
    ppl_c  = sw[str(k_candidate)]['ppl']
    spar_c = sw[str(k_candidate)]['sparsity_pct']
    for k_other in k_vals:
        if k_other == k_candidate:
            continue
        ppl_o  = sw[str(k_other)]['ppl']
        spar_o = sw[str(k_other)]['sparsity_pct']
        if ppl_o < ppl_c and spar_o >= spar_c:
            return False
    return True


print("Dependency bridge ready ✅")
print(f"  STEPS = {STEPS}")
print(f"  SEEDS = {SEEDS}")
print(f"  matplotlib backend : {matplotlib.get_backend()}")
print(f"  is_pareto_optimal  : defined ✅")

Dependency bridge ready ✅
  STEPS = 10000
  SEEDS = [42, 123, 7]
  matplotlib backend : Agg
  is_pareto_optimal  : defined ✅


## 23. GPT2 Tokenizer Experiments

All previous sections used `bert-base-uncased` (vocab=30,522).




> Results saved under `gpt2_*` keys in results.json — fully resumable.

In [9]:
from transformers import GPT2TokenizerFast
from torch.utils.data import Dataset, DataLoader

# ── GPT2 tokenizer ────────────────────────────────────────────
gpt2_tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
GPT2_VOCAB_SIZE = len(gpt2_tokenizer)   # 50257
print(f"GPT2 vocab size: {GPT2_VOCAB_SIZE}")

class GPT2WikiText2Dataset(Dataset):
    def __init__(self, split, seq_len=128):
        raw     = load_dataset("wikitext", "wikitext-2-raw-v1", split=split)
        all_ids = []
        for item in raw:
            t = item["text"].strip()
            if t:
                all_ids.extend(
                    gpt2_tokenizer.encode(t, add_special_tokens=False)
                )
        n = (len(all_ids) // (seq_len + 1)) * (seq_len + 1)
        self.chunks = torch.tensor(all_ids[:n], dtype=torch.long).view(-1, seq_len + 1)
        print(f"  GPT2 WikiText-2 [{split}]: {len(self.chunks)} chunks")
    def __len__(self):        return len(self.chunks)
    def __getitem__(self, i): return self.chunks[i]

print("\nLoading GPT2-tokenized WikiText-2...")
gpt2_train, gpt2_val = make_loaders(
    GPT2WikiText2Dataset, seq_len=SEQ_LEN, batch_size=BATCH_SIZE
)
print(f"  Train batches: {len(gpt2_train)} | Val batches: {len(gpt2_val)}")
print("GPT2 data loaders ready ✅")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

GPT2 vocab size: 50257

Loading GPT2-tokenized WikiText-2...
  GPT2 WikiText-2 [train]: 18194 chunks
  GPT2 WikiText-2 [validation]: 1880 chunks
  Train batches: 568 | Val batches: 58
GPT2 data loaders ready ✅


In [10]:
class GPT2WikiText103Dataset(Dataset):
    def __init__(self, split, seq_len=128, n_chunks=20000):
        raw     = load_dataset("wikitext", "wikitext-103-raw-v1",
                               split=split, trust_remote_code=True)
        all_ids = []
        for item in raw:
            t = item["text"].strip()
            if not t:
                continue
            all_ids.extend(
                gpt2_tokenizer.encode(t, add_special_tokens=False)
            )
            if len(all_ids) >= n_chunks * (seq_len + 1):
                break
        n = (len(all_ids) // (seq_len + 1)) * (seq_len + 1)
        self.chunks = torch.tensor(all_ids[:n], dtype=torch.long).view(-1, seq_len + 1)
        print(f"  GPT2 WikiText-103 [{split}]: {len(self.chunks)} chunks")
    def __len__(self):        return len(self.chunks)
    def __getitem__(self, i): return self.chunks[i]

print("Loading GPT2-tokenized WikiText-103...")
gpt2_wt103_train, gpt2_wt103_val = make_loaders(
    GPT2WikiText103Dataset, seq_len=SEQ_LEN, batch_size=BATCH_SIZE,
    n_chunks=20000
)
print(f"  Train batches: {len(gpt2_wt103_train)} | Val batches: {len(gpt2_wt103_val)}")
print("GPT2 WikiText-103 loaders ready ✅")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikitext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading GPT2-tokenized WikiText-103...


wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikitext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  GPT2 WikiText-103 [train]: 20000 chunks
  GPT2 WikiText-103 [validation]: 1880 chunks
  Train batches: 625 | Val batches: 58
GPT2 WikiText-103 loaders ready ✅


## 24. GPT2 Main Results — Dense vs ADAPT-BIO

 GPT2TokenizerFast (vocab=50,257).
3 seeds × 2 model sizes × 2 datasets = 24 training runs.
Results saved under `gpt2_wt2_128`, `gpt2_wt2_256`, `gpt2_wt103_128`, `gpt2_wt103_256`.

In [11]:
GPT2_STEPS = 10000

def run_seeds_gpt2(ModelClass, label, seeds, loader_train, loader_val, **model_kwargs):
    ppls = []
    for seed in seeds:
        torch.manual_seed(seed); np.random.seed(seed)
        model = ModelClass(**model_kwargs).to(DEVICE)
        ppl, _ = train_run(model, loader_train, loader_val,
                           steps=GPT2_STEPS, label=f"{label}-s{seed}")
        ppls.append(round(ppl, 4))
        del model; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return ppls

# ── WikiText-2 | d_model=128 ──────────────────────────────────
if 'gpt2_wt2_128' not in results:
    print("=" * 55)
    print("  GPT2 | WikiText-2 | d_model=128")
    print("=" * 55)
    dense_ppls = run_seeds_gpt2(
        FairDenseTransformer, "GPT2-Dense-WT2-128", SEEDS,
        gpt2_train, gpt2_val,
        vocab_size=GPT2_VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN)
    adapt_ppls = run_seeds_gpt2(
        ADAPTBIOTransformer, "GPT2-ADAPT-WT2-128", SEEDS,
        gpt2_train, gpt2_val,
        vocab_size=GPT2_VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN, k=7,
        anticipation_steps=100, rna_update_interval=500)
    results['gpt2_wt2_128'] = {
        'dense': {'ppls': dense_ppls,
                  'mean': round(float(np.mean(dense_ppls)), 4),
                  'std':  round(float(np.std(dense_ppls)),  4)},
        'adapt': {'ppls': adapt_ppls,
                  'mean': round(float(np.mean(adapt_ppls)), 4),
                  'std':  round(float(np.std(adapt_ppls)),  4)},
        'flops': theoretical_flops(7, SEQ_LEN),
    }
    save_results()
else:
    print("gpt2_wt2_128 already done — skipping.")

r = results['gpt2_wt2_128']
print(f"\n  GPT2 Dense-WT2-128 : PPL = {r['dense']['mean']:.2f} ± {r['dense']['std']:.2f}")
print(f"  GPT2 ADAPT-WT2-128 : PPL = {r['adapt']['mean']:.2f} ± {r['adapt']['std']:.2f}")

# ── WikiText-2 | d_model=256 ──────────────────────────────────
if 'gpt2_wt2_256' not in results:
    print("\n" + "=" * 55)
    print("  GPT2 | WikiText-2 | d_model=256")
    print("=" * 55)
    dense_ppls = run_seeds_gpt2(
        FairDenseTransformer, "GPT2-Dense-WT2-256", SEEDS,
        gpt2_train, gpt2_val,
        vocab_size=GPT2_VOCAB_SIZE, d_model=256, num_heads=8,
        num_layers=2, seq_len=SEQ_LEN)
    adapt_ppls = run_seeds_gpt2(
        ADAPTBIOTransformer, "GPT2-ADAPT-WT2-256", SEEDS,
        gpt2_train, gpt2_val,
        vocab_size=GPT2_VOCAB_SIZE, d_model=256, num_heads=8,
        num_layers=2, seq_len=SEQ_LEN, k=7,
        anticipation_steps=100, rna_update_interval=500)
    results['gpt2_wt2_256'] = {
        'dense': {'ppls': dense_ppls,
                  'mean': round(float(np.mean(dense_ppls)), 4),
                  'std':  round(float(np.std(dense_ppls)),  4)},
        'adapt': {'ppls': adapt_ppls,
                  'mean': round(float(np.mean(adapt_ppls)), 4),
                  'std':  round(float(np.std(adapt_ppls)),  4)},
        'flops': theoretical_flops(7, SEQ_LEN),
    }
    save_results()
else:
    print("gpt2_wt2_256 already done — skipping.")

r2 = results['gpt2_wt2_256']
print(f"\n  GPT2 Dense-WT2-256 : PPL = {r2['dense']['mean']:.2f} ± {r2['dense']['std']:.2f}")
print(f"  GPT2 ADAPT-WT2-256 : PPL = {r2['adapt']['mean']:.2f} ± {r2['adapt']['std']:.2f}")

# ── WikiText-103 | d_model=128 ────────────────────────────────
if 'gpt2_wt103_128' not in results:
    print("\n" + "=" * 55)
    print("  GPT2 | WikiText-103 | d_model=128")
    print("=" * 55)
    dense_ppls = run_seeds_gpt2(
        FairDenseTransformer, "GPT2-Dense-103-128", SEEDS,
        gpt2_wt103_train, gpt2_wt103_val,
        vocab_size=GPT2_VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN)
    adapt_ppls = run_seeds_gpt2(
        ADAPTBIOTransformer, "GPT2-ADAPT-103-128", SEEDS,
        gpt2_wt103_train, gpt2_wt103_val,
        vocab_size=GPT2_VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN, k=7,
        anticipation_steps=100, rna_update_interval=500)
    results['gpt2_wt103_128'] = {
        'dense': {'ppls': dense_ppls,
                  'mean': round(float(np.mean(dense_ppls)), 4),
                  'std':  round(float(np.std(dense_ppls)),  4)},
        'adapt': {'ppls': adapt_ppls,
                  'mean': round(float(np.mean(adapt_ppls)), 4),
                  'std':  round(float(np.std(adapt_ppls)),  4)},
        'flops': theoretical_flops(7, SEQ_LEN),
    }
    save_results()
else:
    print("gpt2_wt103_128 already done — skipping.")

r3 = results['gpt2_wt103_128']
print(f"\n  GPT2 Dense-103-128 : PPL = {r3['dense']['mean']:.2f} ± {r3['dense']['std']:.2f}")
print(f"  GPT2 ADAPT-103-128 : PPL = {r3['adapt']['mean']:.2f} ± {r3['adapt']['std']:.2f}")

# ── WikiText-103 | d_model=256 ────────────────────────────────
if 'gpt2_wt103_256' not in results:
    print("\n" + "=" * 55)
    print("  GPT2 | WikiText-103 | d_model=256")
    print("=" * 55)
    dense_ppls = run_seeds_gpt2(
        FairDenseTransformer, "GPT2-Dense-103-256", SEEDS,
        gpt2_wt103_train, gpt2_wt103_val,
        vocab_size=GPT2_VOCAB_SIZE, d_model=256, num_heads=8,
        num_layers=2, seq_len=SEQ_LEN)
    adapt_ppls = run_seeds_gpt2(
        ADAPTBIOTransformer, "GPT2-ADAPT-103-256", SEEDS,
        gpt2_wt103_train, gpt2_wt103_val,
        vocab_size=GPT2_VOCAB_SIZE, d_model=256, num_heads=8,
        num_layers=2, seq_len=SEQ_LEN, k=7,
        anticipation_steps=100, rna_update_interval=500)
    results['gpt2_wt103_256'] = {
        'dense': {'ppls': dense_ppls,
                  'mean': round(float(np.mean(dense_ppls)), 4),
                  'std':  round(float(np.std(dense_ppls)),  4)},
        'adapt': {'ppls': adapt_ppls,
                  'mean': round(float(np.mean(adapt_ppls)), 4),
                  'std':  round(float(np.std(adapt_ppls)),  4)},
        'flops': theoretical_flops(7, SEQ_LEN),
    }
    save_results()
else:
    print("gpt2_wt103_256 already done — skipping.")

r4 = results['gpt2_wt103_256']
print(f"\n  GPT2 Dense-103-256 : PPL = {r4['dense']['mean']:.2f} ± {r4['dense']['std']:.2f}")
print(f"  GPT2 ADAPT-103-256 : PPL = {r4['adapt']['mean']:.2f} ± {r4['adapt']['std']:.2f}")

# ── Git commit ────────────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: GPT2 main results — WT-2 + WT-103, d=128 + d=256"')
import subprocess
result = subprocess.run(
    f"cd /kaggle/working/adapt-bio && git push "
    f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main",
    shell=True, capture_output=True, text=True
)
print(result.stdout); print(result.stderr)
print("✅ Pushed to GitHub")

  GPT2 | WikiText-2 | d_model=128
  [GPT2-Dense-WT2-128-s42] step     0/10000 | loss=11.0256 | elapsed=0s
  [GPT2-Dense-WT2-128-s42] step  1000/10000 | loss=7.1059 | elapsed=85s
  [GPT2-Dense-WT2-128-s42] step  2000/10000 | loss=6.7766 | elapsed=178s
  [GPT2-Dense-WT2-128-s42] step  3000/10000 | loss=6.4980 | elapsed=272s
  [GPT2-Dense-WT2-128-s42] step  4000/10000 | loss=6.4585 | elapsed=366s
  [GPT2-Dense-WT2-128-s42] step  5000/10000 | loss=6.2041 | elapsed=460s
  [GPT2-Dense-WT2-128-s42] step  6000/10000 | loss=6.2707 | elapsed=554s
  [GPT2-Dense-WT2-128-s42] step  7000/10000 | loss=6.2985 | elapsed=648s
  [GPT2-Dense-WT2-128-s42] step  8000/10000 | loss=6.2443 | elapsed=742s
  [GPT2-Dense-WT2-128-s42] step  9000/10000 | loss=6.3003 | elapsed=836s
  [GPT2-Dense-WT2-128-s42] ✅ Final Val PPL = 561.2910  (total 932s)
  [GPT2-Dense-WT2-128-s123] step     0/10000 | loss=11.0370 | elapsed=0s
  [GPT2-Dense-WT2-128-s123] step  1000/10000 | loss=7.0257 | elapsed=93s
  [GPT2-Dense-WT2-128-s1

## 25. GPT2 T=512 Scaling Experiment — WikiText-2

Same protocol as Section 10 (BERT T=512, WikiText-2), now with GPT2 tokenizer.  
Dense vs ADAPT-BIO at sequence length T=512 on WikiText-2 (GPT2-tokenized).  
Results saved under `gpt2_wt2_t512`.

The key scaling claim: FLOPs advantage grows quadratically with T.  
At T=128: 18.3× reduction. At T=512: 73.1× reduction.

In [12]:
SEQ_LEN_512      = 512
BATCH_SIZE_512   = 8
STEPS_512_GPT2   = 3000   # fewer steps — T=512 batches are ~4× more expensive

if 'gpt2_wt2_t512' not in results:
    print("Loading GPT2 WikiText-2 T=512 data...")
    gpt2_train_512, gpt2_val_512 = make_loaders(
        GPT2WikiText2Dataset, seq_len=SEQ_LEN_512, batch_size=BATCH_SIZE_512
    )
    print(f"  Train batches: {len(gpt2_train_512)} | Val batches: {len(gpt2_val_512)}")

    gpt2_t512_results = {}

    # ── Dense @ T=512 ─────────────────────────────────────────
    torch.manual_seed(42); np.random.seed(42)
    model_dense = FairDenseTransformer(
        vocab_size=GPT2_VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN_512, dropout=0.3
    ).to(DEVICE)
    ppl_d, time_d = train_run(model_dense, gpt2_train_512, gpt2_val_512,
                               steps=STEPS_512_GPT2, label="GPT2-Dense-T512")
    gpt2_t512_results['dense'] = {
        'ppl':         round(ppl_d, 4),
        'wall_time_s': round(time_d, 1),
        'flops_ratio': 1.0,
        'dense_attn_flops': SEQ_LEN_512 * SEQ_LEN_512,
    }
    del model_dense; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    # ── ADAPT-BIO @ T=512 ─────────────────────────────────────
    torch.manual_seed(42); np.random.seed(42)
    model_adapt = ADAPTBIOTransformer(
        vocab_size=GPT2_VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN_512, k=7,
        anticipation_steps=100, rna_update_interval=500
    ).to(DEVICE)
    ppl_a, time_a = train_run(model_adapt, gpt2_train_512, gpt2_val_512,
                               steps=STEPS_512_GPT2, label="GPT2-ADAPT-T512")
    gpt2_t512_results['adapt'] = {
        'ppl':         round(ppl_a, 4),
        'wall_time_s': round(time_a, 1),
        'flops':       theoretical_flops(7, SEQ_LEN_512),
        'flops_ratio': round(SEQ_LEN_512 / 7, 1),
    }
    del model_adapt; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    results['gpt2_wt2_t512'] = gpt2_t512_results
    save_results()
else:
    print("gpt2_wt2_t512 already done — skipping.")

# ── Print comparison table ────────────────────────────────────
r_d = results['gpt2_wt2_t512']['dense']
r_a = results['gpt2_wt2_t512']['adapt']

dense_attn  = SEQ_LEN_512 * SEQ_LEN_512
adapt_attn  = SEQ_LEN_512 * 7
flops_ratio = dense_attn / adapt_attn

print(f"\n  GPT2 WikiText-2 | T=512 Sequence Length Experiment")
print(f"  {'Metric':<22} | {'Dense':>10} | {'ADAPT-BIO':>10}")
print("  " + "-" * 48)
print(f"  {'Val PPL ↓':<22} | {r_d['ppl']:>10.2f} | {r_a['ppl']:>10.2f}")
print(f"  {'Wall time (s) ↓':<22} | {r_d['wall_time_s']:>10.1f} | {r_a['wall_time_s']:>10.1f}")
print(f"  {'Dense attn FLOPs':<22} | {dense_attn:>10,} | {adapt_attn:>10,}")
print(f"  {'FLOPs reduction':<22} | {'1.0×':>10} | {flops_ratio:>9.1f}×")
print(f"\n  T=128 FLOPs ratio : 18.3×")
print(f"  T=512 FLOPs ratio : {flops_ratio:.1f}×  "
      f"({'%.1f' % (flops_ratio / 18.3)}× larger advantage at longer sequence)")

# ── Figure 16 ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

t_vals  = [64, 128, 256, 512, 1024]
ratios  = [t / 7 for t in t_vals]
axes[0].plot(t_vals, ratios, 'o-', color='#2196F3', lw=2.5, ms=8)
axes[0].axvline(x=128, color='gray', ls='--', lw=1.2, label='T=128 (main exp)')
axes[0].axvline(x=512, color='red',  ls='--', lw=1.2, label='T=512 (this exp)')
axes[0].set_xlabel('Sequence Length T', fontsize=11)
axes[0].set_ylabel('FLOPs Reduction (×) ↑', fontsize=11)
axes[0].set_title('(A) FLOPs Advantage vs T', fontweight='bold')
axes[0].legend(fontsize=9)
for t, r in zip(t_vals, ratios):
    axes[0].annotate(f'{r:.0f}×', (t, r), textcoords='offset points',
                     xytext=(5, 5), fontsize=9)

models  = ['Dense\n(T=512)', 'ADAPT-BIO\n(T=512)']
ppls    = [r_d['ppl'], r_a['ppl']]
colors  = ['#78909C', '#4CAF50']
axes[1].bar(models, ppls, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
for i, v in enumerate(ppls):
    axes[1].text(i, v + 1, f'{v:.2f}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Validation Perplexity ↓', fontsize=11)
axes[1].set_title('(B) Val PPL at T=512', fontweight='bold')

times  = [r_d['wall_time_s'], r_a['wall_time_s']]
axes[2].bar(models, times, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
for i, v in enumerate(times):
    axes[2].text(i, v + 1, f'{v:.0f}s', ha='center', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Wall Time (s) ↓', fontsize=11)
axes[2].set_title('(C) Training Time at T=512', fontweight='bold')

fig.suptitle(f'Figure 16: GPT2 WikiText-2 T=512 Sequence Length Scaling\n'
             f'ADAPT-BIO FLOPs advantage: 18.3× (T=128) → {flops_ratio:.1f}× (T=512)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/adapt-bio/paper/figures/gpt2_wt2_t512_experiment.png',
            dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Figure 16 saved → paper/figures/gpt2_wt2_t512_experiment.png")

os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          f'"results: GPT2 WikiText-2 T=512 scaling + Figure 16 [{flops_ratio:.0f}x FLOPs at T=512]"')
import subprocess
result = subprocess.run(
    f"cd /kaggle/working/adapt-bio && git push "
    f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main",
    shell=True, capture_output=True, text=True
)
print(result.stdout); print(result.stderr)
print("✅ Pushed to GitHub")

Loading GPT2 WikiText-2 T=512 data...
  GPT2 WikiText-2 [train]: 4575 chunks
  GPT2 WikiText-2 [validation]: 472 chunks
  Train batches: 571 | Val batches: 59
  [GPT2-Dense-T512] step     0/3000 | loss=11.0447 | elapsed=0s
  [GPT2-Dense-T512] step  1000/3000 | loss=7.1328 | elapsed=97s
  [GPT2-Dense-T512] step  2000/3000 | loss=7.0259 | elapsed=193s
  [GPT2-Dense-T512] ✅ Final Val PPL = 950.9047  (total 292s)
  [GPT2-ADAPT-T512] step     0/3000 | loss=11.0039 | elapsed=0s
  [GPT2-ADAPT-T512] step  1000/3000 | loss=6.9564 | elapsed=97s
  [GPT2-ADAPT-T512] step  2000/3000 | loss=6.7312 | elapsed=195s
  [GPT2-ADAPT-T512] ✅ Final Val PPL = 931.9236  (total 294s)
✅ results.json saved (['main_wt2_128', 'main_wt2_256', 'k_sweep', 'pareto_analysis', 'rna_sweep', 'component_ablation', 't512_experiment', 'benchmarks', 'main_wt103_128', 'main_wt103_256', 'significance_tests', 'wt103_k_sweep', 'wt103_pareto_analysis', 'wt103_rna_sweep', 'wt103_component_ablation', 'wt103_t512_experiment', 'wt103_b

## 26. GPT2 T=512 Scaling Experiment — WikiText-103

Same protocol as Section 19 (BERT T=512, WikiText-103), now with GPT2 tokenizer.  
Dense vs ADAPT-BIO at sequence length T=512 on WikiText-103 (GPT2-tokenized).  
Results saved under `gpt2_wt103_t512`.

In [13]:
SEQ_LEN_512         = 512
BATCH_SIZE_512      = 8
STEPS_512_GPT2_103  = 3000   # T=512 batches are ~4× more expensive

if 'gpt2_wt103_t512' not in results:
    print("Loading GPT2 WikiText-103 T=512 data...")
    gpt2_wt103_train_512, gpt2_wt103_val_512 = make_loaders(
        GPT2WikiText103Dataset, seq_len=SEQ_LEN_512,
        batch_size=BATCH_SIZE_512, n_chunks=20000
    )
    print(f"  Train batches: {len(gpt2_wt103_train_512)} | Val batches: {len(gpt2_wt103_val_512)}")

    gpt2_t512_103_results = {}

    # ── Dense @ T=512 ─────────────────────────────────────────
    torch.manual_seed(42); np.random.seed(42)
    model_dense = FairDenseTransformer(
        vocab_size=GPT2_VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN_512, dropout=0.3
    ).to(DEVICE)
    ppl_d, time_d = train_run(model_dense, gpt2_wt103_train_512, gpt2_wt103_val_512,
                               steps=STEPS_512_GPT2_103, label="GPT2-WT103-Dense-T512")
    gpt2_t512_103_results['dense'] = {
        'ppl':         round(ppl_d, 4),
        'wall_time_s': round(time_d, 1),
        'flops_ratio': 1.0,
        'dense_attn_flops': SEQ_LEN_512 * SEQ_LEN_512,
    }
    del model_dense; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    # ── ADAPT-BIO @ T=512 ─────────────────────────────────────
    torch.manual_seed(42); np.random.seed(42)
    model_adapt = ADAPTBIOTransformer(
        vocab_size=GPT2_VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN_512, k=7,
        anticipation_steps=100, rna_update_interval=500
    ).to(DEVICE)
    ppl_a, time_a = train_run(model_adapt, gpt2_wt103_train_512, gpt2_wt103_val_512,
                               steps=STEPS_512_GPT2_103, label="GPT2-WT103-ADAPT-T512")
    gpt2_t512_103_results['adapt'] = {
        'ppl':         round(ppl_a, 4),
        'wall_time_s': round(time_a, 1),
        'flops':       theoretical_flops(7, SEQ_LEN_512),
        'flops_ratio': round(SEQ_LEN_512 / 7, 1),
    }
    del model_adapt; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    results['gpt2_wt103_t512'] = gpt2_t512_103_results
    save_results()
else:
    print("gpt2_wt103_t512 already done — skipping.")

# ── Print comparison ──────────────────────────────────────────
r_d = results['gpt2_wt103_t512']['dense']
r_a = results['gpt2_wt103_t512']['adapt']
flops_ratio_512 = SEQ_LEN_512 / 7

print(f"\n  GPT2 WikiText-103 | T=512 Experiment")
print(f"  {'Metric':<22} | {'Dense':>10} | {'ADAPT-BIO':>10}")
print("  " + "-" * 48)
print(f"  {'Val PPL ↓':<22} | {r_d['ppl']:>10.2f} | {r_a['ppl']:>10.2f}")
print(f"  {'Wall time (s) ↓':<22} | {r_d['wall_time_s']:>10.1f} | {r_a['wall_time_s']:>10.1f}")
print(f"  {'FLOPs reduction':<22} | {'1.0×':>10} | {flops_ratio_512:>9.1f}×")
print(f"\n  T=128 FLOPs ratio : 18.3×")
print(f"  T=512 FLOPs ratio : {flops_ratio_512:.1f}×")

# ── Figure 17 ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

models  = ['Dense\n(T=512)', 'ADAPT-BIO\n(T=512)']
ppls    = [r_d['ppl'], r_a['ppl']]
times   = [r_d['wall_time_s'], r_a['wall_time_s']]
colors  = ['#78909C', '#4CAF50']

axes[0].bar(models, ppls, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
for i, v in enumerate(ppls):
    axes[0].text(i, v + 1, f'{v:.2f}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Validation Perplexity ↓', fontsize=11)
axes[0].set_title('(A) Val PPL at T=512', fontweight='bold')

axes[1].bar(models, times, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
for i, v in enumerate(times):
    axes[1].text(i, v + 1, f'{v:.0f}s', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Wall Time (s) ↓', fontsize=11)
axes[1].set_title('(B) Training Time at T=512', fontweight='bold')

fig.suptitle(f'Figure 17: GPT2 WikiText-103 T=512 Sequence Length Scaling\n'
             f'FLOPs advantage: 18.3× (T=128) → {flops_ratio_512:.1f}× (T=512)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/adapt-bio/paper/figures/gpt2_wt103_t512_experiment.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 17 saved → paper/figures/gpt2_wt103_t512_experiment.png")

os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: GPT2 WikiText-103 T=512 scaling + Figure 17"')
import subprocess
result = subprocess.run(
    f"cd /kaggle/working/adapt-bio && git push "
    f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main",
    shell=True, capture_output=True, text=True
)
print(result.stdout); print(result.stderr)
print("✅ Pushed to GitHub")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikitext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading GPT2 WikiText-103 T=512 data...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikitext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  GPT2 WikiText-103 [train]: 20000 chunks
  GPT2 WikiText-103 [validation]: 472 chunks
  Train batches: 2500 | Val batches: 59
  [GPT2-WT103-Dense-T512] step     0/3000 | loss=11.0705 | elapsed=0s
  [GPT2-WT103-Dense-T512] step  1000/3000 | loss=7.0991 | elapsed=97s
  [GPT2-WT103-Dense-T512] step  2000/3000 | loss=6.8007 | elapsed=193s
  [GPT2-WT103-Dense-T512] ✅ Final Val PPL = 925.7437  (total 292s)
  [GPT2-WT103-ADAPT-T512] step     0/3000 | loss=10.9995 | elapsed=0s
  [GPT2-WT103-ADAPT-T512] step  1000/3000 | loss=6.9347 | elapsed=98s
  [GPT2-WT103-ADAPT-T512] step  2000/3000 | loss=6.9231 | elapsed=195s
  [GPT2-WT103-ADAPT-T512] ✅ Final Val PPL = 903.8804  (total 294s)
✅ results.json saved (['main_wt2_128', 'main_wt2_256', 'k_sweep', 'pareto_analysis', 'rna_sweep', 'component_ablation', 't512_experiment', 'benchmarks', 'main_wt103_128', 'main_wt103_256', 'significance_tests', 'wt103_k_sweep', 'wt103_pareto_analysis', 'wt103_rna_sweep', 'wt103_component_ablation', 'wt103_t512_exper

## 27. GPT2 k-Sweep & RNA Interval Sweep

Reproduces the original paper's k-sweep (k=7 → PPL=53.03) and RNA sweep
(Δ=500 → PPL=29.41 beats Frozen 35.20) with GPT2 tokenizer on WikiText-2.
Results saved under `gpt2_k_sweep` and `gpt2_rna_sweep`.

In [14]:
K_VALS = [3, 5, 7, 9, 12]

# ── GPT2 k-sweep ──────────────────────────────────────────────
if 'gpt2_k_sweep' not in results:
    print("Running GPT2 k-sweep (5 runs × 10000 steps)...")
    gpt2_k_results = {}
    for kv in K_VALS:
        torch.manual_seed(42); np.random.seed(42)
        model = ADAPTBIOTransformer(
            vocab_size=GPT2_VOCAB_SIZE, d_model=128, num_heads=4,
            num_layers=2, seq_len=SEQ_LEN, k=kv,
            anticipation_steps=100, rna_update_interval=500
        ).to(DEVICE)
        ppl, _ = train_run(model, gpt2_train, gpt2_val,
                           steps=GPT2_STEPS, label=f"GPT2-k={kv}")
        flops = theoretical_flops(kv, SEQ_LEN)
        gpt2_k_results[str(kv)] = {
            'ppl':          round(ppl, 4),
            'sparsity_pct': round(flops['flop_reduction'] * 100, 2),
            'flops_ratio':  round(flops['flops_ratio'], 2),
        }
        del model; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    results['gpt2_k_sweep'] = gpt2_k_results
    save_results()
else:
    print("gpt2_k_sweep already done — skipping.")

print(f"\n  GPT2 k-Sweep (WikiText-2):")
print(f"  {'k':>4} | {'Val PPL':>8} | {'Sparsity':>9} | {'FLOPs ratio':>11}")
print("  " + "-"*40)
for kv in K_VALS:
    r = results['gpt2_k_sweep'][str(kv)]
    chosen_k_gpt2 = results.get('gpt2_pareto_analysis', {}).get('chosen_k')
    marker = " ← chosen operating point" if kv == chosen_k_gpt2 else ""
    print(f"  {kv:>4} | {r['ppl']:>8.2f} | {r['sparsity_pct']:>8.1f}% | {r['flops_ratio']:>9.1f}×{marker}")

# ── GPT2 RNA sweep ────────────────────────────────────────────
DELTA_VALS = [50, 200, 500, 2000]

if 'gpt2_rna_sweep' not in results:
    print("\nRunning GPT2 RNA sweep (5 runs × 10000 steps)...")
    gpt2_rna_results = {}

    for delta in DELTA_VALS:
        torch.manual_seed(42); np.random.seed(42)
        model = ADAPTBIOTransformer(
            vocab_size=GPT2_VOCAB_SIZE, d_model=128, num_heads=4,
            num_layers=2, seq_len=SEQ_LEN, k=7,
            anticipation_steps=100, rna_update_interval=delta
        ).to(DEVICE)
        ppl, _ = train_run(model, gpt2_train, gpt2_val,
                           steps=GPT2_STEPS, label=f"GPT2-Δ={delta}")
        gpt2_rna_results[str(delta)] = round(ppl, 4)
        del model; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    # Frozen baseline
    torch.manual_seed(42); np.random.seed(42)
    model = ADAPTBIOTransformer(
        vocab_size=GPT2_VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN, k=7,
        anticipation_steps=100, rna_update_interval=999999,
        use_rna=False
    ).to(DEVICE)
    ppl, _ = train_run(model, gpt2_train, gpt2_val,
                       steps=GPT2_STEPS, label="GPT2-Frozen")
    gpt2_rna_results['frozen'] = round(ppl, 4)
    del model; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    results['gpt2_rna_sweep'] = gpt2_rna_results
    save_results()
else:
    print("gpt2_rna_sweep already done — skipping.")

gpt2_frozen = results['gpt2_rna_sweep']['frozen']
print(f"\n  GPT2 RNA Sweep (WikiText-2):")
print(f"  {'Δ (steps)':>12} | {'Val PPL':>8} | Note")
print("  " + "-"*45)
for delta in DELTA_VALS:
    ppl = results['gpt2_rna_sweep'][str(delta)]
    beat = "✅ beats frozen" if ppl < gpt2_frozen else "❌ worse than frozen"
    print(f"  {delta:>12} | {ppl:>8.2f} | {beat}")
print(f"  {'Frozen':>12} | {gpt2_frozen:>8.2f} | static baseline")

# ── Figures ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A — k-sweep
ppls_k = [results['gpt2_k_sweep'][str(k)]['ppl'] for k in K_VALS]
spar_k = [results['gpt2_k_sweep'][str(k)]['sparsity_pct'] for k in K_VALS]
ax1 = axes[0]; ax2 = ax1.twinx()
ax1.plot(K_VALS, ppls_k, 'o-', color='#2196F3', lw=2.5, ms=8, label='Val PPL ↓')
ax1.axvline(x=7, color='red', ls='--', lw=1.5, label='k=7 operating point')
ax2.plot(K_VALS, spar_k, 's--', color='#4CAF50', lw=2, ms=8, label='Sparsity %')
ax1.set_xlabel('k (neighbours per token)', fontsize=11)
ax1.set_ylabel('Val PPL ↓', color='#2196F3', fontsize=11)
ax2.set_ylabel('Sparsity % ↑', color='#4CAF50', fontsize=11)
ax2.set_ylim(85, 100)
lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, lbl1+lbl2, fontsize=9)
ax1.set_title('(A) GPT2 k-Sweep — WikiText-2', fontweight='bold')
ax1.set_xticks(K_VALS)

# Panel B — RNA sweep
x_labels = [str(d) for d in DELTA_VALS] + ['Frozen']
y_vals   = [results['gpt2_rna_sweep'][str(d)] for d in DELTA_VALS] + [gpt2_frozen]
colors   = ['#FF7043' if v >= gpt2_frozen else '#4CAF50' for v in y_vals[:-1]] + ['#9E9E9E']
axes[1].bar(x_labels, y_vals, color=colors, edgecolor='black', linewidth=0.8, width=0.6)
axes[1].axhline(y=gpt2_frozen, color='black', ls='--', lw=1.5,
                label=f'Frozen ({gpt2_frozen:.2f})')
for i, v in enumerate(y_vals):
    axes[1].text(i, v + max(y_vals)*0.01, f'{v:.1f}',
                 ha='center', fontsize=9, fontweight='bold')
axes[1].set_xlabel('Δ (steps)', fontsize=11)
axes[1].set_ylabel('Val PPL ↓', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].set_title('(B) GPT2 RNA Sweep — WikiText-2', fontweight='bold')

fig.suptitle('Figure 15: GPT2 Tokenizer — k-Sweep & RNA Sweep\n'
             'WikiText-2, d_model=128, 10000 steps',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/adapt-bio/paper/figures/gpt2_sweeps.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 15 saved → paper/figures/gpt2_sweeps.png")
# ── Pareto frontier analysis — GPT2 tokenizer ────────────────
sweep = results['gpt2_k_sweep']
K_VALS_P = sorted(int(k) for k in sweep.keys())

pareto_ks   = [k for k in K_VALS_P if is_pareto_optimal(k, sweep, K_VALS_P)]
chosen_k    = max(pareto_ks, key=lambda k: sweep[str(k)]['sparsity_pct'])
chosen_ppl  = sweep[str(chosen_k)]['ppl']
chosen_spar = sweep[str(chosen_k)]['sparsity_pct']

print("=" * 60)
print("  PARETO FRONTIER ANALYSIS — GPT2 k-sweep")
print("=" * 60)
print(f"\n  Pareto-optimal k values : {pareto_ks}")
print(f"  Chosen operating point  : k={chosen_k} "
      f"(PPL={chosen_ppl:.2f}, Sparsity={chosen_spar:.1f}%)\n")

print(f"  {'k':>4} | {'PPL':>8} | {'Sparsity':>9} | {'Status':>25}")
print("  " + "-" * 52)
for kv in K_VALS_P:
    r = sweep[str(kv)]
    if kv == chosen_k:
        status = "← chosen operating point"
    elif kv in pareto_ks:
        status = "Pareto-optimal"
    else:
        status = "dominated"
    print(f"  {kv:>4} | {r['ppl']:>8.2f} | {r['sparsity_pct']:>8.1f}% | {status:>25}")

dominated_by = [
    kv for kv in K_VALS_P if kv != chosen_k
    and sweep[str(kv)]['ppl'] < chosen_ppl
    and sweep[str(kv)]['sparsity_pct'] >= chosen_spar
]

if dominated_by:
    print(f"\n  ⚠️  k={dominated_by} dominates chosen k={chosen_k} "
          f"— operating point needs revision.")
else:
    print(f"\n  ✅ k={chosen_k} is Pareto-optimal under GPT2 tokenizer.")

# Cross-check against BERT results for both datasets
bert_wt2_k   = results.get('pareto_analysis', {}).get('chosen_k')
bert_wt103_k = results.get('wt103_pareto_analysis', {}).get('chosen_k')

print(f"\n  Cross-tokenizer consistency check:")
print(f"  {'Source':<25} | {'Chosen k':>8} | {'Agrees with GPT2?':>18}")
print("  " + "-" * 56)
for source, ref_k in [("BERT WT-2",   bert_wt2_k),
                       ("BERT WT-103", bert_wt103_k)]:
    if ref_k is not None:
        agrees = "✅ yes" if ref_k == chosen_k else "⚠️  no"
        print(f"  {source:<25} | {ref_k:>8} | {agrees:>18}")

all_agree = all(
    ref == chosen_k for ref in [bert_wt2_k, bert_wt103_k]
    if ref is not None
)
if all_agree:
    print(f"\n  ✅ k={chosen_k} is the Pareto-optimal operating point "
          f"across all tokenizers and datasets — paper claim is fully supported.")
else:
    print(f"\n  ⚠️  Chosen k is not consistent across all configs — "
          f"paper claim needs qualification.")

results['gpt2_pareto_analysis'] = {
    'pareto_ks':       pareto_ks,
    'chosen_k':        chosen_k,
    'chosen_ppl':      chosen_ppl,
    'chosen_spar':     chosen_spar,
    'dominated_by':    dominated_by,
    'pareto_optimal':  len(dominated_by) == 0,
    'agrees_with_bert_wt2':   chosen_k == bert_wt2_k   if bert_wt2_k   else None,
    'agrees_with_bert_wt103': chosen_k == bert_wt103_k if bert_wt103_k else None,
    'consistent_across_all':  all_agree,
}
save_results()
print("\n✅ GPT2 Pareto analysis saved to results.json")
# ── Git commit ────────────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: GPT2 k-sweep + RNA sweep + Figure 15"')
import subprocess
result = subprocess.run(
    f"cd /kaggle/working/adapt-bio && git push "
    f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main",
    shell=True, capture_output=True, text=True
)
print(result.stdout); print(result.stderr)
print("✅ Pushed to GitHub")

Running GPT2 k-sweep (5 runs × 10000 steps)...
  [GPT2-k=3] step     0/10000 | loss=11.0031 | elapsed=0s
  [GPT2-k=3] step  1000/10000 | loss=7.0115 | elapsed=88s
  [GPT2-k=3] step  2000/10000 | loss=6.6538 | elapsed=175s
  [GPT2-k=3] step  3000/10000 | loss=6.4740 | elapsed=263s
  [GPT2-k=3] step  4000/10000 | loss=6.1205 | elapsed=351s
  [GPT2-k=3] step  5000/10000 | loss=6.1623 | elapsed=439s
  [GPT2-k=3] step  6000/10000 | loss=6.1686 | elapsed=527s
  [GPT2-k=3] step  7000/10000 | loss=6.0240 | elapsed=615s
  [GPT2-k=3] step  8000/10000 | loss=5.9000 | elapsed=703s
  [GPT2-k=3] step  9000/10000 | loss=5.8848 | elapsed=791s
  [GPT2-k=3] ✅ Final Val PPL = 611.8960  (total 881s)
  [GPT2-k=5] step     0/10000 | loss=11.0031 | elapsed=0s
  [GPT2-k=5] step  1000/10000 | loss=7.0120 | elapsed=88s
  [GPT2-k=5] step  2000/10000 | loss=6.6504 | elapsed=175s
  [GPT2-k=5] step  3000/10000 | loss=6.4720 | elapsed=263s
  [GPT2-k=5] step  4000/10000 | loss=6.1191 | elapsed=351s
  [GPT2-k=5] step 

## 28. GPT2 Cross-Tokenizer Summary & Conclusions

Consolidates all GPT2 results and compares against BERT tokenizer findings.

In [15]:
# ── Cross-tokenizer comparison table ─────────────────────────
print("=" * 70)
print("  CROSS-TOKENIZER COMPARISON — BERT vs GPT2")
print("=" * 70)

print(f"\n  {'Config':<28} | {'BERT PPL':>10} | {'GPT2 PPL':>10} | {'Consistent?':>12}")
print("  " + "-"*68)

comparisons = [
    ("WT-2 d=128",
     results['main_wt2_128']['dense']['mean'],
     results['gpt2_wt2_128']['dense']['mean'],
     results['main_wt2_128']['adapt']['mean'],
     results['gpt2_wt2_128']['adapt']['mean']),

    ("WT-2 d=256",
     results['main_wt2_256']['dense']['mean'],
     results['gpt2_wt2_256']['dense']['mean'],
     results['main_wt2_256']['adapt']['mean'],
     results['gpt2_wt2_256']['adapt']['mean']),

    ("WT-103 d=128",
     results['main_wt103_128']['dense']['mean'],
     results['gpt2_wt103_128']['dense']['mean'],
     results['main_wt103_128']['adapt']['mean'],
     results['gpt2_wt103_128']['adapt']['mean']),

    ("WT-103 d=256",
     results['main_wt103_256']['dense']['mean'],
     results['gpt2_wt103_256']['dense']['mean'],
     results['main_wt103_256']['adapt']['mean'],
     results['gpt2_wt103_256']['adapt']['mean']),
]

for label, bert_dense, gpt2_dense, bert_adapt, gpt2_adapt in comparisons:
    bert_adapt_wins = bert_adapt < bert_dense
    gpt2_adapt_wins = gpt2_adapt < gpt2_dense
    consistent = "✅ both"      if (bert_adapt_wins and gpt2_adapt_wins) else \
                 "⚠️ BERT only" if bert_adapt_wins else \
                 "⚠️ GPT2 only" if gpt2_adapt_wins else "❌ neither"
    print(f"  {'Dense ' + label:<28} | {bert_dense:>10.2f} | {gpt2_dense:>10.2f} | {'(baseline)':>12}")
    print(f"  {'ADAPT ' + label:<28} | {bert_adapt:>10.2f} | {gpt2_adapt:>10.2f} | {consistent:>12}")
    print()

# ── k-sweep cross-tokenizer — fully data-driven ───────────────
print(f"\n  k-Sweep: Cross-tokenizer Pareto analysis")
print(f"  {'k':>4} | {'BERT PPL':>10} | {'GPT2 PPL':>10} | {'BERT Sparsity':>14} | {'Status':>25}")
print("  " + "-"*70)

bert_sweep = results['k_sweep']
gpt2_sweep = results['gpt2_k_sweep']
K_VALS_CT  = sorted(int(k) for k in bert_sweep.keys())

def is_cross_pareto_optimal(k_candidate, bert_sw, gpt2_sw, k_vals):
    """
    k_candidate is cross-tokenizer Pareto-optimal if no other k achieves
    BOTH lower PPL under BOTH tokenizers AND higher sparsity simultaneously.
    """
    ppl_b_c  = bert_sw[str(k_candidate)]['ppl']
    ppl_g_c  = gpt2_sw[str(k_candidate)]['ppl']
    spar_c   = bert_sw[str(k_candidate)]['sparsity_pct']  # sparsity is tokenizer-independent
    for k_other in k_vals:
        if k_other == k_candidate:
            continue
        ppl_b_o = bert_sw[str(k_other)]['ppl']
        ppl_g_o = gpt2_sw[str(k_other)]['ppl']
        spar_o  = bert_sw[str(k_other)]['sparsity_pct']
        # dominates if better PPL under both tokenizers AND higher or equal sparsity
        if ppl_b_o < ppl_b_c and ppl_g_o < ppl_g_c and spar_o >= spar_c:
            return False
    return True

cross_pareto_ks = [
    k for k in K_VALS_CT
    if is_cross_pareto_optimal(k, bert_sweep, gpt2_sweep, K_VALS_CT)
]

# Chosen operating point: highest sparsity among cross-Pareto-optimal k values
chosen_k = max(cross_pareto_ks,
               key=lambda k: bert_sweep[str(k)]['sparsity_pct'])

for kv in K_VALS_CT:
    ppl_b = bert_sweep[str(kv)]['ppl']
    ppl_g = gpt2_sweep[str(kv)]['ppl']
    spar  = bert_sweep[str(kv)]['sparsity_pct']
    if kv == chosen_k:
        status = "← chosen (cross-Pareto optimal)"
    elif kv in cross_pareto_ks:
        status = "Pareto-optimal"
    else:
        status = "dominated"
    print(f"  {kv:>4} | {ppl_b:>10.2f} | {ppl_g:>10.2f} | {spar:>13.1f}% | {status:>25}")

print(f"\n  Cross-Pareto-optimal k values : {cross_pareto_ks}")
print(f"  Chosen operating point        : k={chosen_k}")

if chosen_k != results.get('pareto_analysis', {}).get('chosen_k'):
    print(f"\n  ⚠️  Cross-tokenizer optimal k={chosen_k} differs from "
          f"single-tokenizer result — investigate before finalising paper claim.")
else:
    print(f"\n  ✅ Cross-tokenizer Pareto result agrees with single-tokenizer analysis.")

# ── RNA sweep cross-tokenizer ─────────────────────────────────
bert_frozen = results['rna_sweep']['frozen']
gpt2_frozen = results['gpt2_rna_sweep']['frozen']
delta_vals  = sorted(int(d) for d in results['rna_sweep'].keys() if d != 'frozen')

print(f"\n  RNA Sweep: Does optimal Δ beat Frozen under both tokenizers?")
print(f"  {'Δ':>8} | {'BERT PPL':>10} | {'GPT2 PPL':>10} | {'Both beat frozen?':>18}")
print("  " + "-"*52)

best_delta      = None
best_delta_ppl  = float('inf')

for d in delta_vals:
    b = results['rna_sweep'][str(d)]
    g = results['gpt2_rna_sweep'][str(d)]
    both_beat = "✅ yes"        if (b < bert_frozen and g < gpt2_frozen) else \
                "⚠️ BERT only" if b < bert_frozen else \
                "⚠️ GPT2 only" if g < gpt2_frozen else "❌ neither"
    print(f"  {d:>8} | {b:>10.2f} | {g:>10.2f} | {both_beat:>18}")
    # Track which delta gives lowest average PPL across both tokenizers
    avg_ppl = (b + g) / 2
    if avg_ppl < best_delta_ppl:
        best_delta_ppl = avg_ppl
        best_delta     = d

print(f"  {'Frozen':>8} | {bert_frozen:>10.2f} | {gpt2_frozen:>10.2f} | {'(baseline)':>18}")
print(f"\n  Δ with lowest avg PPL across both tokenizers: Δ={best_delta}")

# Compute single-tokenizer BERT optimal delta inline — no stored key needed
best_delta_bert = min(delta_vals, key=lambda d: results['rna_sweep'][str(d)])
if best_delta != best_delta_bert:
    print(f"  ⚠️  Cross-tokenizer optimal Δ={best_delta} differs from "
          f"single-tokenizer BERT optimal Δ={best_delta_bert} — "
          f"verify before finalising paper claim.")
else:
    print(f"  ✅ Cross-tokenizer optimal Δ={best_delta} agrees with single-tokenizer result.")

# Save cross-tokenizer analysis
results['cross_tokenizer_analysis'] = {
    'cross_pareto_ks':  cross_pareto_ks,
    'chosen_k':         chosen_k,
    'best_delta':       best_delta,
    'best_delta_ppl':   round(best_delta_ppl, 4),
}
save_results()
print("\n✅ Cross-tokenizer analysis saved to results.json")

  CROSS-TOKENIZER COMPARISON — BERT vs GPT2

  Config                       |   BERT PPL |   GPT2 PPL |  Consistent?
  --------------------------------------------------------------------
  Dense WT-2 d=128             |     490.57 |     562.21 |   (baseline)
  ADAPT WT-2 d=128             |     532.98 |     606.76 |    ❌ neither

  Dense WT-2 d=256             |     360.35 |     421.98 |   (baseline)
  ADAPT WT-2 d=256             |     482.96 |     550.27 |    ❌ neither

  Dense WT-103 d=128           |     485.59 |     559.07 |   (baseline)
  ADAPT WT-103 d=128           |     523.78 |     598.33 |    ❌ neither

  Dense WT-103 d=256           |     354.89 |     415.52 |   (baseline)
  ADAPT WT-103 d=256           |     467.84 |     531.75 |    ❌ neither


  k-Sweep: Cross-tokenizer Pareto analysis
     k |   BERT PPL |   GPT2 PPL |  BERT Sparsity |                    Status
  ----------------------------------------------------------------------
     3 |     538.61 |     611.90 |   

In [16]:
save_results()

print("=" * 60)
print("  ADAPT-BIO — Notebook 2 Complete")
print("=" * 60)
print(f"  Total experiment keys : {len(results)}")
print(f"  Keys: {list(results.keys())}")
print(f"\n  GPT2 figures saved:")
print(f"    paper/figures/gpt2_sweeps.png")
print(f"    paper/figures/gpt2_wt2_t512_experiment.png")
print(f"    paper/figures/gpt2_wt103_t512_experiment.png")
print(f"\n  GitHub: https://github.com/Kritika11052005/adapt-bio")

os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"notebook: complete ADAPT-BIO Notebook 2 — GPT2 tokenizer experiments, '
          'T=512 WT-2 + WT-103, k-sweep, RNA sweep, cross-tokenizer summary"')
import subprocess
result = subprocess.run(
    f"cd /kaggle/working/adapt-bio && git push "
    f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main",
    shell=True, capture_output=True, text=True
)
print(result.stdout); print(result.stderr)
print("\n✅ ADAPT-BIO Notebook 2 fully complete and committed to GitHub.")

✅ results.json saved (['main_wt2_128', 'main_wt2_256', 'k_sweep', 'pareto_analysis', 'rna_sweep', 'component_ablation', 't512_experiment', 'benchmarks', 'main_wt103_128', 'main_wt103_256', 'significance_tests', 'wt103_k_sweep', 'wt103_pareto_analysis', 'wt103_rna_sweep', 'wt103_component_ablation', 'wt103_t512_experiment', 'wt103_benchmarks', 'gpt2_wt2_128', 'gpt2_wt2_256', 'gpt2_wt103_128', 'gpt2_wt103_256', 'gpt2_wt2_t512', 'gpt2_wt103_t512', 'gpt2_k_sweep', 'gpt2_rna_sweep', 'gpt2_pareto_analysis', 'cross_tokenizer_analysis'])
  ADAPT-BIO — Notebook 2 Complete
  Total experiment keys : 27
  Keys: ['main_wt2_128', 'main_wt2_256', 'k_sweep', 'pareto_analysis', 'rna_sweep', 'component_ablation', 't512_experiment', 'benchmarks', 'main_wt103_128', 'main_wt103_256', 'significance_tests', 'wt103_k_sweep', 'wt103_pareto_analysis', 'wt103_rna_sweep', 'wt103_component_ablation', 'wt103_t512_experiment', 'wt103_benchmarks', 'gpt2_wt2_128', 'gpt2_wt2_256', 'gpt2_wt103_128', 'gpt2_wt103_256', 'g